# 09. 오류 전파와 종료 상태 확인


## Goal

set -e의 조건부 호출 예외를 재현하고, stdout·stderr·종료 상태를 독립적으로 검증한다. [교안 09-1](../../09-testing-debugging/09-1-errors-tracing.md)과 연결된다.


## Setup

새 임시 폴더에서 합성 출력만 사용한다. 예상 실패는 if로 포착하여 노트북 전체 실행은 성공하도록 구성한다.


In [ ]:
from pathlib import Path
import os
import shutil
import tempfile

lab_dir = Path(tempfile.mkdtemp(prefix="bash-book-09-"))
os.environ["BASH_LAB_DIR"] = str(lab_dir)
print(f"새 임시 실습 디렉터리: {lab_dir}")


## Steps

### 1. 조건에서 호출된 함수

예상: false 이후에도 continued와 reported success가 출력된다. -e가 모든 문맥에서 동작하지 않음을 확인한다.


In [ ]:
%%bash
set -u
observed=$(bash -e -c 'work() { false; printf "continued\n"; }; if work; then printf "reported success\n"; fi')
printf '%s\n' "$observed"
[[ $observed == $'continued\nreported success' ]]


### 2. 명시적 실패 전달

예상: 실패 상태를 return으로 전달하므로 failure만 출력된다.


In [ ]:
%%bash
set -u
observed=$(bash -e -c 'work() { false || return 1; printf "unreachable\n"; }; if work; then printf "success\n"; else printf "failure\n"; fi')
printf '%s\n' "$observed"
[[ $observed == failure ]]


### 3. 정상 출력·오류 출력·종료 상태 검사


In [ ]:
%%bash
set -u
status=0
bash -c 'printf "data\n"; printf "diagnostic\n" >&2; exit 7' > "$BASH_LAB_DIR/out" 2> "$BASH_LAB_DIR/err" || status=$?
[[ $status == 7 ]] || exit 1
[[ $(< "$BASH_LAB_DIR/out") == data ]] || exit 1
[[ $(< "$BASH_LAB_DIR/err") == diagnostic ]] || exit 1
printf 'stdout=data stderr=diagnostic status=%s\n' "$status"


## Checks

- 조건부 호출에서 -e만 의존한 결과와 명시적 return 결과가 다른가?
- 상태 7을 오류 출력과 별도로 검증했는가?
- exit 7을 exit 3으로 바꾸면 마지막 검사가 실패하는가? 원래 코드로 복구하고 재실행한다.


## Next Steps

검증할 함수를 라이브러리로 분리하고 source 시 부수 효과가 없는지 확인한다.


In [ ]:
import shutil
from pathlib import Path
import os

lab_dir = Path(os.environ["BASH_LAB_DIR"])
shutil.rmtree(lab_dir, ignore_errors=True)
print(f"정리 완료: {lab_dir}")
